# 06 — Model Comparison

Benchmarks both tracks: **MPJPE** (mean per-joint position error) and
**inference time**.

### Why there are no RMSE columns

Earlier versions reported `RMSE-kin` and `RMSE-MP` for the depth comparison. Both are invalid
and have been removed.

`evaluation/kinematics_ground_truth.py::sample_ground_truth()` builds its
ground truth by drawing **random joint angles** from `JOINT_LIMITS` and running
forward kinematics. It never reads the recorded data, so sample *i* of the
ground truth has no relationship to frame *i* of the video — the metric
compares two unrelated point clouds. Measured on the arm track:

| Predictions fed in | RMSE-kin |
|---|---|
| Real | 1206.009 |
| Shuffled | 1205.255 |
| All zeros | **122.861** |

Shuffling the predictions — destroying every correspondence — moves the score
by 0.06%, and a model that predicts nothing scores **ten times better** than
the trained LSTM. A metric that rewards predicting nothing measures nothing.

`RMSE-MP` was separately circular: `mp = gt*scale + noise` with both injected
by that same script, so it only ever measured its own noise model — which is
why it was identical across all three models.

**MPJPE is genuine** (`preds` vs `y`, both from the real recordings) and is
what these tables report.


In [1]:
import os, sys
_SCRIPTS = os.path.abspath('..')
sys.path.insert(0, _SCRIPTS)

import numpy as np
from evaluation.benchmark_models import mpjpe_mm, measure_inference_ms
from models.lstm_predictor import LSTMPredictor
from models.gru_predictor import GRUPredictor
from models.transformer_predictor import TransformerPredictor

Classes = {'LSTM': LSTMPredictor, 'GRU': GRUPredictor, 'Transformer': TransformerPredictor}

def run_track(data_path, model_paths, feature_dim):
    # wrist_idx / seed are gone with the RMSE columns: they only fed the
    # kinematics reconstruction, whose ground truth was random robot poses
    # unrelated to this data. See the notes at the top.
    d = np.load(data_path)
    X, y = d['X'].astype(np.float32), d['y'].astype(np.float32)
    n = len(X)
    sample_seq = X[0]

    results = {}
    for name, path in model_paths.items():
        m = Classes[name](window_size=30, feature_dim=feature_dim)
        m.load(path)
        preds = np.array([m.predict(X[i]) for i in range(n)], dtype=np.float32)
        mpjpe = mpjpe_mm(preds, y, feature_dim=feature_dim)
        timing = measure_inference_ms(m, sample_seq)


        results[name] = dict(mpjpe=mpjpe, inf_ms=timing['mean_ms'])
    return results


def print_table(title, results):
    print(f'=== {title} ===')
    print(f'{"Model":<12} {"MPJPE(mm)":>10} {"Inf(ms)":>9}')
    for name, r in results.items():
        print(f'{name:<12} {r["mpjpe"]:>10.3f} {r["inf_ms"]:>9.3f}')
    print()


## Hand track (wrist = landmark 0)

In [2]:
hand_models = {
    'LSTM':        '../models/lstm_predictor.pt',
    'GRU':         '../models/gru_predictor.pt',
    'Transformer': '../models/transformer_predictor.pt',
}
hand_results = run_track('../dataset.npz', hand_models, feature_dim=63)
print_table('HAND TRACK', hand_results)


[LSTMPredictor] loaded ../models/lstm_predictor.pt
[GRUPredictor] loaded ../models/gru_predictor.pt


/home/user_02/Desktop/ros2_ws/src/robot_arm/scripts/models/transformer_predictor.py:68: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


[TransformerPredictor] loaded ../models/transformer_predictor.pt
=== HAND TRACK ===
Model         MPJPE(mm)   Inf(ms)
LSTM           2579.166     0.381
GRU            2513.406     0.272
Transformer    2550.099     0.591



## Arm track (wrist = landmark 16, `R_WRIST`)

In [3]:
pose_models = {
    'LSTM':        '../models/pose_lstm_predictor.pt',
    'GRU':         '../models/pose_gru_predictor.pt',
    'Transformer': '../models/pose_transformer_predictor.pt',
}
pose_results = run_track('../pose_dataset.npz', pose_models, feature_dim=99)
print_table('ARM TRACK', pose_results)


[LSTMPredictor] loaded ../models/pose_lstm_predictor.pt
[GRUPredictor] loaded ../models/pose_gru_predictor.pt
[TransformerPredictor] loaded ../models/pose_transformer_predictor.pt
=== ARM TRACK ===
Model         MPJPE(mm)   Inf(ms)
LSTM           1683.295     0.378
GRU            1713.581     0.272
Transformer    1556.181     0.601

